# Project Codebase Notebook

This notebook is the unified execution flow for this project.

## Pipeline
1. Config
2. Prepare Data
3. Build Index
4. Inference
5. Evaluation

## Usage Notes
- Execute cells from top to bottom.
- Keep input and output paths consistent with this repository.
- This notebook uses native components (`sentence-transformers`, `faiss`, `transformers`) without LangChain wrappers.

# 1. Config

This section defines project paths, model names, retrieval and generation hyperparameters, and runtime device selection (CUDA/MPS/CPU).

Run this once before all later stages.

In [6]:
from pathlib import Path
import torch

# ===== Paths =====
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
INDEX_DIR = BASE_DIR / "index_store"
OUTPUT_DIR = BASE_DIR / "outputs"

CORPUS_PATH = DATA_DIR / "corpus.jsonl"
BENCHMARK_PATH = DATA_DIR / "benchmark.json"
TEST_QUERIES_PATH = DATA_DIR / "test_queries.json"

INDEX_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Models =====
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# ===== Baseline settings =====
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 5
RETRIEVAL_CANDIDATE_K = 12
RETRIEVAL_ALPHA = 0.75
MAX_NEW_TOKENS = 128

# ===== Runtime =====
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Initialize System: Running on {DEVICE.upper()} mode")

TEMPERATURE = 0.0

# ===== Files =====
FAISS_INDEX_PATH = str(INDEX_DIR / "faiss_index")
INFERENCE_OUTPUT_PATH = OUTPUT_DIR / "inference_output.json"
EVAL_OUTPUT_PATH = OUTPUT_DIR / "evaluation_results.json"

Initialize System: Running on CUDA mode


# 2. Prepare Data

This stage runs data preprocessing to produce the three runtime files used by later steps:
- `data/corpus.jsonl`
- `data/benchmark.json`
- `data/test_queries.json`

Run this stage after config and before indexing.

In [7]:
import runpy
from pathlib import Path

prepare_script = Path("prepare_data.py")
if not prepare_script.exists():
    raise FileNotFoundError(f"Missing {prepare_script}")

runpy.run_path(str(prepare_script), run_name="__main__")
print("Prepare stage complete.")

Data ready: corpus.jsonl is physically chunked and benchmark.json uses keyword matching!
Prepare stage complete.


# 3. Build Index

This stage reads `data/corpus.jsonl`, computes normalized dense embeddings, and builds a FAISS inner-product index.

Outputs:
- `index_store/faiss_index/index.faiss`
- `index_store/faiss_index/index_metadata.json`

In [8]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


def indexing() -> None:
    corpus = load_jsonl(CORPUS_PATH)
    if not corpus:
        raise ValueError(f"Empty corpus: {CORPUS_PATH}")

    texts = []
    metadata_rows = []

    for row in corpus:
        chunk_id = row["doc_id"]
        title = row.get("title", "")
        source = row.get("source", "")
        document_type = row.get("document_type", "")
        text = row["text"]

        enriched_chunk = f"Title: {title}\nSource: {source}\nType: {document_type}\n\n{text}"
        texts.append(enriched_chunk)
        metadata_rows.append({
            "doc_id": chunk_id,
            "title": title,
            "source": source,
            "document_type": document_type,
            "chunk_id": chunk_id,
            "text": enriched_chunk,
        })

    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
    embeddings = embedder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    index_dir = Path(FAISS_INDEX_PATH)
    index_dir.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(index_dir / "index.faiss"))
    save_json(metadata_rows, index_dir / "index_metadata.json")

    print(f"Built FAISS index with {len(metadata_rows)} chunks.")
    print(f"Saved to: {index_dir}")


indexing()

Batches: 100%|█████████████████████████████████████████████████████████████████████████| 94/94 [00:08<00:00, 10.91it/s]

Built FAISS index with 2999 chunks.
Saved to: index_store\faiss_index


# Shared Utilities

Reusable helper functions used by indexing, inference, and evaluation are centralized here:
- JSON/JSONL read-write
- text normalization and overlap scoring
- EM/F1 and reciprocal-rank metrics
- prompt construction

In [9]:
import json
import re
from pathlib import Path
from typing import List, Dict, Any


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text


def exact_match(pred: str, gold: str) -> int:
    return int(normalize_text(pred) == normalize_text(gold))


def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()

    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0

    pred_counts = {}
    for t in pred_tokens:
        pred_counts[t] = pred_counts.get(t, 0) + 1

    gold_counts = {}
    for t in gold_tokens:
        gold_counts[t] = gold_counts.get(t, 0) + 1

    common = 0
    for token, count in pred_counts.items():
        if token in gold_counts:
            common += min(count, gold_counts[token])

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def lexical_overlap_score(query: str, text: str) -> float:
    query_tokens = set(normalize_text(query).split())
    if not query_tokens:
        return 0.0
    text_tokens = set(normalize_text(text).split())
    if not text_tokens:
        return 0.0
    return len(query_tokens.intersection(text_tokens)) / len(query_tokens)


def reciprocal_rank(retrieved_doc_ids: List[str], gold_doc_ids: set[str]) -> float:
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id in gold_doc_ids:
            return 1.0 / rank
    return 0.0


def build_prompt(query: str, retrieved_chunks: List[str]) -> str:
    context = "\n\n".join(
        [f"[Chunk {i+1}]\n{chunk}" for i, chunk in enumerate(retrieved_chunks)]
    )
    return f"""You are a culinary question-answering assistant specialised in cuisine.
Use only the provided context to answer.
If the answer is not in the context, output exactly: "I cannot determine this from the provided context."
Keep the answer concise (1-2 sentences), factual, and avoid adding assumptions.

Context:
{context}

Question:
{query}

Final Answer:"""

# 4. Inference

This stage loads the FAISS index and metadata, performs dense retrieval with lexical fusion reranking, then generates answers with the configured causal LM.

Output:
- `outputs/inference_output.json`

In [10]:
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_vector_index():
    index_dir = Path(FAISS_INDEX_PATH)
    index = faiss.read_index(str(index_dir / "index.faiss"))
    metadata_rows = load_json(index_dir / "index_metadata.json")
    return index, metadata_rows


def load_embedder():
    return SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)


def load_generator():
    tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        GENERATION_MODEL_NAME,
        torch_dtype=torch.float32 if DEVICE == "cpu" else torch.float16,
    )
    if DEVICE == "cuda":
        model = model.to("cuda")
    model.eval()
    return tokenizer, model


def generate_answer(tokenizer, model, prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if TEMPERATURE > 0:
        gen_kwargs.update({"do_sample": True, "temperature": TEMPERATURE})
    else:
        gen_kwargs.update({"do_sample": False})

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "Final Answer:" in decoded:
        return decoded.split("Final Answer:", 1)[-1].strip()
    if "Answer:" in decoded:
        return decoded.split("Answer:", 1)[-1].strip()
    return decoded.strip().split("\n")[0]


def hybrid_retrieve(index, metadata_rows, embedder, query: str):
    query_vec = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, idxs = index.search(query_vec, RETRIEVAL_CANDIDATE_K)
    scores = scores[0]
    idxs = idxs[0]

    candidates = []
    for idx, score in zip(idxs, scores):
        if idx < 0 or idx >= len(metadata_rows):
            continue
        row = metadata_rows[int(idx)]
        text = row["text"]
        candidates.append({
            "text": text,
            "metadata": {
                "doc_id": row["doc_id"],
                "title": row.get("title", ""),
                "source": row.get("source", ""),
                "document_type": row.get("document_type", ""),
                "chunk_id": row.get("chunk_id", row["doc_id"]),
            },
            "semantic_raw": float(score),
        })

    if not candidates:
        return []

    sem_vals = [c["semantic_raw"] for c in candidates]
    sem_min = min(sem_vals)
    sem_max = max(sem_vals)

    reranked = []
    for c in candidates:
        if sem_max == sem_min:
            semantic = 1.0
        else:
            semantic = (c["semantic_raw"] - sem_min) / (sem_max - sem_min)
        lexical = lexical_overlap_score(query, c["text"])
        hybrid = RETRIEVAL_ALPHA * semantic + (1.0 - RETRIEVAL_ALPHA) * lexical
        reranked.append((c, hybrid, semantic, lexical))

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked[:TOP_K]


def main() -> None:
    queries = load_json(TEST_QUERIES_PATH)
    index, metadata_rows = load_vector_index()
    embedder = load_embedder()
    tokenizer, model = load_generator()

    outputs = []
    for item in queries:
        qid = item["id"]
        query = item["query"]

        reranked = hybrid_retrieve(index, metadata_rows, embedder, query)
        retrieved_chunks = [x[0]["text"] for x in reranked]

        prompt = build_prompt(query, retrieved_chunks)
        answer = generate_answer(tokenizer, model, prompt)

        outputs.append(
            {
                "id": qid,
                "query": query,
                "answer": answer,
                "retrieved_chunks": [
                    {
                        "text": x[0]["text"],
                        "metadata": x[0]["metadata"],
                        "hybrid_score": round(x[1], 4),
                        "semantic_score": round(x[2], 4),
                        "lexical_score": round(x[3], 4),
                    }
                    for x in reranked
                ],
            }
        )

        print(f"[{qid}] {query}")
        print(f"Answer: {answer}")
        print("-" * 60)

    save_json(outputs, INFERENCE_OUTPUT_PATH)
    print(f"Saved inference outputs to {INFERENCE_OUTPUT_PATH}")


main()

C:\Users\25247\anaconda3\envs\trim_rag\lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


[1] What is the difference between dim sum and yum cha?
Answer: The main difference between dim sum and yum cha lies in their origin and cultural significance. Dim sum originated in China and refers to a meal consisting of small portions of food served in bamboo steamers or on small plates, typically including dumplings. On the other hand, yum cha originates in Hong Kong and refers to the entire occasion of eating dim sum, often from circling push carts, along with drinking copious amounts of tea over a social and leisurely brunch. Both dishes are enjoyed by families and friends, but they serve different purposes and have distinct cultural meanings.
------------------------------------------------------------
[2] What does the word bento mean and what does a bento box typically do?
Answer: The word "bento" refers to a variety of food items, commonly including meat/fish, rice, and salad, kept separate in a container. A bento box is a common feature in Japanese cuisine, particularly in f

# 5. Evaluation

This stage compares predictions against benchmark labels and reports retrieval and generation metrics.

Output:
- `outputs/evaluation_results.json`

In [11]:
def evaluate_main() -> None:
    benchmark = load_json(BENCHMARK_PATH)
    predictions = load_json(INFERENCE_OUTPUT_PATH)

    pred_map = {x["id"]: x for x in predictions}

    results = []
    retrieval_hits = []
    retrieval_recalls = []
    retrieval_precisions = []
    retrieval_mrr = []
    em_scores = []
    f1_scores = []

    for ex in benchmark:
        qid = ex["id"]
        gold_answer = ex["gold_answer"]
        gold_doc_ids = set(ex["gold_evidence_doc_ids"])

        if qid not in pred_map:
            continue

        pred = pred_map[qid]
        pred_answer = pred["answer"]

        retrieved_doc_ids_list = [chunk["metadata"]["doc_id"] for chunk in pred["retrieved_chunks"]]
        retrieved_doc_ids = set(retrieved_doc_ids_list)

        overlap = len(gold_doc_ids.intersection(retrieved_doc_ids))
        hit_at_5 = int(overlap > 0)
        recall_at_5 = overlap / len(gold_doc_ids) if gold_doc_ids else 0.0
        precision_at_5 = overlap / len(retrieved_doc_ids) if retrieved_doc_ids else 0.0
        mrr_at_5 = reciprocal_rank(retrieved_doc_ids_list, gold_doc_ids)
        em = exact_match(pred_answer, gold_answer)
        f1 = token_f1(pred_answer, gold_answer)

        retrieval_hits.append(hit_at_5)
        retrieval_recalls.append(recall_at_5)
        retrieval_precisions.append(precision_at_5)
        retrieval_mrr.append(mrr_at_5)
        em_scores.append(em)
        f1_scores.append(f1)

        results.append(
            {
                "id": qid,
                "query": ex["query"],
                "gold_answer": gold_answer,
                "pred_answer": pred_answer,
                "hit@5": hit_at_5,
                "recall@5": round(recall_at_5, 4),
                "precision@5": round(precision_at_5, 4),
                "mrr@5": round(mrr_at_5, 4),
                "exact_match": em,
                "token_f1": round(f1, 4),
            }
        )

    summary = {
        "num_examples": len(results),
        "retrieval_hit@5": round(sum(retrieval_hits) / len(retrieval_hits), 4) if retrieval_hits else 0.0,
        "retrieval_recall@5": round(sum(retrieval_recalls) / len(retrieval_recalls), 4) if retrieval_recalls else 0.0,
        "retrieval_precision@5": round(sum(retrieval_precisions) / len(retrieval_precisions), 4) if retrieval_precisions else 0.0,
        "retrieval_mrr@5": round(sum(retrieval_mrr) / len(retrieval_mrr), 4) if retrieval_mrr else 0.0,
        "generation_exact_match": round(sum(em_scores) / len(em_scores), 4) if em_scores else 0.0,
        "generation_token_f1": round(sum(f1_scores) / len(f1_scores), 4) if f1_scores else 0.0,
        "details": results,
    }

    save_json(summary, EVAL_OUTPUT_PATH)
    print("Evaluation summary:")
    print(summary)
    print(f"Saved evaluation results to {EVAL_OUTPUT_PATH}")


evaluate_main()

Evaluation summary:
{'num_examples': 30, 'retrieval_hit@5': 0.6667, 'retrieval_recall@5': 0.6667, 'retrieval_precision@5': 0.1333, 'retrieval_mrr@5': 0.5556, 'generation_exact_match': 0.0, 'generation_token_f1': 0.2628, 'details': [{'id': '1', 'query': 'What is the difference between dim sum and yum cha?', 'gold_answer': 'Dim sum refers to small portions of food served in steamers or on plates, while yum cha is the whole occasion of eating dim sum together with tea—yum cha means drink tea in Cantonese.', 'pred_answer': 'The main difference between dim sum and yum cha lies in their origin and cultural significance. Dim sum originated in China and refers to a meal consisting of small portions of food served in bamboo steamers or on small plates, typically including dumplings. On the other hand, yum cha originates in Hong Kong and refers to the entire occasion of eating dim sum, often from circling push carts, along with drinking copious amounts of tea over a social and leisurely brunch. 